# Gymnasium agent — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-arena/competition-baseline/blob/main/gymnasium/agent_baseline.ipynb)

A minimal, runnable **single-agent RL** baseline for any ml-arena Gymnasium
competition (LunarLander, CartPole, the Atari suite, the MuJoCo control tasks, ...).
It defines a random-action `Agent`, deploys it, and shows the leaderboard.

> Open in Colab, run the cells top to bottom. Edit **two** things: your API token
> and `COMPETITION_ID` (each competition page shows its id in the URL). The default
> is `43` — **LunarLander-v3**.

The agent is deliberately trivial — it plays *random* actions. Beating it is the
whole point: replace `choose_action` with a policy you have trained.

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk==0.3.0" numpy   # installs the `mlarena` package

import mlarena

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page -> API Keys)
COMPETITION_ID = 43

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Define your agent

The worker imports the symbol **`Agent`** from your `agent.py` and instantiates it
as `Agent(observation_space, action_space)` (both are Gymnasium `Space` objects).
Every step it calls `choose_action(...)` and expects an action from `action_space`.
Return `None` once the episode is over (`terminated` or `truncated`).

Keep the class **self-contained** — it is uploaded on its own, so it may not import
the environment or other local files (only public packages available in the runtime).

In [ ]:
import numpy as np


class Agent:
    def __init__(self, observation_space, action_space):
        self.observation_space = observation_space
        self.action_space = action_space
        self._rng = np.random.default_rng(0)

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        # TODO: replace this random policy with your trained model.
        return int(self.action_space.sample())

## 2. (optional) Sanity-check it locally

The real Gymnasium spaces come from the environment at run time; here we just fake a
discrete action space to confirm `choose_action` returns a valid action.

In [ ]:
class _FakeDiscrete:
    def __init__(self, n): self.n = n
    def sample(self): return int(np.random.randint(self.n))

agent = Agent(observation_space=None, action_space=_FakeDiscrete(4))
print('sample action:', agent.choose_action(observation=[0.0] * 8))

## 3. Submit

This uploads the class as `agent.py`, deploys it, and streams status until it scores.

In [ ]:
# Uploads the Agent class source as `agent.py`, then creates + deploys the attachment.
result = client.submit(competition_id=COMPETITION_ID, agent=Agent)
print(result)

# Poll until the run reaches a terminal state (deploy -> queue -> run -> scored).
for line in client.tail_logs(COMPETITION_ID, result["attache_agent_id"]):
    print(line)

## 4. Leaderboard

In [ ]:
client.leaderboard(COMPETITION_ID)

## 5. Where to go from here

The random agent is just a launch pad. To climb the leaderboard:

- **Train a policy offline** (Stable-Baselines3 / CleanRL: PPO, DQN, SAC), save the
  weights, and load them in `__init__` — upload the weights file alongside `agent.py`
  with `client.submit(COMPETITION_ID, files=["agent.py", "model.zip"])`.
- **Match the runtime** to your framework with
  `client.submit(..., runtime={"language": "python", "framework": "torch"})`.
- **Reuse this notebook** for any single-agent Gymnasium/Atari/MuJoCo competition —
  only `COMPETITION_ID` changes; the `Agent` contract is identical.